In [1]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

vehicle_types = {
    "Truck": {"capacity": (3000, 12000), "fuel": (2.5, 5)},
    "Van": {"capacity": (500, 2000), "fuel": (8, 14)},
    "Mini Truck": {"capacity": (1000, 3500), "fuel": (5, 10)},
    "Bus": {"capacity": (0, 0), "fuel": (2, 4)}
}

rows = []

for i in range(1000):

    vehicle_type = random.choice(list(vehicle_types.keys()))

    age = random.randint(1, 12)

    odometer = random.randint(
        age * 20000,
        age * 50000
    )

    trips = random.randint(
        age * 80,
        age * 350
    )

    fuel_efficiency = round(
        random.uniform(
            vehicle_types[vehicle_type]["fuel"][0],
            vehicle_types[vehicle_type]["fuel"][1]
        ),
        2
    )

    maintenance_count = max(
        0,
        int(
            age * 0.8 +
            odometer / 70000 +
            np.random.normal(0, 1)
        )
    )

    avg_load = random.randint(
        30,
        100
    )

    avg_trip_distance = random.randint(
        20,
        600
    )

    engine_temperature = round(
        np.random.normal(85 + age * 1.5, 4),
        1
    )

    tire_health = max(
        30,
        min(
            100,
            int(
                100 -
                age * 5 -
                maintenance_count * 2 +
                np.random.normal(0, 5)
            )
        )
    )

    # Maintenance Risk Score

    risk = 0

    if age > 7:
        risk += 2

    if odometer > 250000:
        risk += 3

    if maintenance_count > 6:
        risk += 2

    if tire_health < 60:
        risk += 2

    if engine_temperature > 95:
        risk += 2

    if avg_load > 90:
        risk += 1

    if risk >= 7:
        breakdown = 1
    elif risk >= 5:
        breakdown = np.random.choice([0,1], p=[0.35,0.65])
    elif risk >= 3:
        breakdown = np.random.choice([0,1], p=[0.75,0.25])
    else:
        breakdown = 0

    rows.append({

        "Vehicle_ID": f"VH{1000+i}",

        "Vehicle_Type": vehicle_type,

        "Age_Years": age,

        "Odometer_km": odometer,

        "Trips_Completed": trips,

        "Fuel_Efficiency_kmpl": fuel_efficiency,

        "Maintenance_Count": maintenance_count,

        "Average_Load_Percent": avg_load,

        "Average_Trip_Distance_km": avg_trip_distance,

        "Engine_Temperature": engine_temperature,

        "Tire_Health": tire_health,

        "Breakdown": breakdown
    })

df = pd.DataFrame(rows)

df.to_csv("fleet_dataset.csv", index=False)

print(df.head())

print("\nDataset Shape:", df.shape)

  Vehicle_ID Vehicle_Type  Age_Years  Odometer_km  Trips_Completed  \
0     VH1000        Truck          1        44299              220   
1     VH1001        Truck         10       421208              930   
2     VH1002        Truck          9       284248             2952   
3     VH1003        Truck          3       115392              588   
4     VH1004        Truck          2        64898              259   

   Fuel_Efficiency_kmpl  Maintenance_Count  Average_Load_Percent  \
0                  3.11                  1                    47   
1                  2.57                 15                    57   
2                  3.55                 12                    87   
3                  3.19                  4                    57   
4                  3.40                  2                    74   

   Average_Trip_Distance_km  Engine_Temperature  Tire_Health  Breakdown  
0                       124                85.9           96          0  
1                     

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("fleet_dataset.csv")

X = df.drop(["Vehicle_ID","Vehicle_Type","Breakdown"], axis=1)
y = df["Breakdown"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.975
